In [12]:
import pandas as pd, geopandas as gpd, duckdb, statsmodels, sklearn, shap
print("All good")

All good


In [2]:
from pathlib import Path
import geopandas as gpd

In [3]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

In [4]:
health_regions = gpd.read_file(RAW / "boundaries" / "HR_000b25a_e" / "HR_000b25a_e.shp")
print(health_regions.shape)
print(health_regions.columns.tolist())

(91, 5)
['HR_UID', 'ENGNAME', 'FRENAME', 'DGUID', 'geometry']


In [5]:
# has 91 health regions
health_regions.drop(columns="geometry").head(10) # drop geometry to make it readable

,HR_UID,ENGNAME,FRENAME,DGUID
0,1020,Eastern Urban Zone,Eastern Urban Zone,2025A00071020
1,1021,Eastern Rural Zone,Eastern Rural Zone,2025A00071021
2,1022,Central Zone,Central Zone,2025A00071022
3,1023,Western Zone,Western Zone,2025A00071023
4,1024,Labrador-Grenfell Zone,Labrador-Grenfell Zone,2025A00071024
5,1100,Prince Edward Island,Île-du-Prince-Édouard,2025A00071100
6,1201,Zone 1 - Western,Zone 1 - Western,2025A00071201
7,1202,Zone 2 - Northern,Zone 2 - Northern,2025A00071202
8,1203,Zone 3 - Eastern,Zone 3 - Eastern,2025A00071203
9,1204,Zone 4 - Central,Zone 4 - Central,2025A00071204


The first two digits are the provinces- 10 = Newfoundland and Labrador, 11 = PEI, 12 = Nova Scotia, 48 = Alberta

In [8]:
alberta = health_regions[health_regions["HR_UID"].astype(str).str.startswith("48")]
alberta.drop(columns="geometry")

,HR_UID,ENGNAME,FRENAME,DGUID
76,4841,Southwest Corridor,Southwest Corridor,2025A00074841
77,4842,Southeast Corridor,Southeast Corridor,2025A00074842
78,4843,Calgary Corridor,Calgary Corridor,2025A00074843
79,4844,Central Corridor,Central Corridor,2025A00074844
80,4845,Edmonton Corridor,Edmonton Corridor,2025A00074845
81,4846,Northwest Corridor,Northwest Corridor,2025A00074846
82,4847,Northeast Corridor,Northeast Corridor,2025A00074847


In [9]:
health_regions["province_code"] = health_regions["HR_UID"].astype(str).str[:2]
health_regions.groupby("province_code").size()

province_code
10     5
11     1
12     4
13     7
24    18
35    29
46     5
47     7
48     7
59     5
60     1
61     1
62     1
dtype: int64

Ontario (35) has the most regions- 29 which is 32% of the total regions. It can dominate clustering and regression if it had data that behaved differently than the rest of the country. We will watch out for that. Some regions are huge such as the PEI (11) so it might behave as an outlier.

In [10]:
health_regions[health_regions["province_code"] == "35"][["HR_UID", "ENGNAME"]].head(10)

,HR_UID,ENGNAME
35,3526,The District of Algoma Health Unit
36,3530,Durham Regional Health Unit
37,3533,Grey Bruce Health Unit
38,3536,Halton Regional Health Unit
39,3537,City of Hamilton Health Unit
40,3540,Chatham-Kent Health Unit
41,3542,Lambton Health Unit
42,3544,Middlesex-London Health Unit
43,3546,Niagara Regional Area Health Unit
44,3547,North Bay Parry Sound District Health Unit


In [17]:
odhf = pd.read_csv(next((RAW / "odhf").rglob("*.csv")), encoding="cp1252")
print(odhf.shape)
print(odhf.columns.tolist())
# odhf- list of about 7,000 healthcare facilities across Canada, with their names, types, and map coordinates.

(7033, 17)
['index', 'facility_name', 'source_facility_type', 'odhf_facility_type', 'provider', 'unit', 'street_no', 'street_name', 'postal_code', 'city', 'province', 'source_format_str_address', 'CSDname', 'CSDuid', 'Pruid', 'latitude', 'longitude']


In [18]:
print(odhf["odhf_facility_type"].unique()) #facility types
odhf[["facility_name", "city", "Pruid", "latitude", "longitude"]].head() # how the data looks

<StringArray>
[        'Ambulatory health care services',
                               'Hospitals',
 'Nursing and residential care facilities',
 'nursing and residential care facilities']
Length: 4, dtype: str


,facility_name,city,Pruid,latitude,longitude
0,Advanced Facial & Nasal Surgery Centre,edmonton,48,NaN,NaN
1,Alberta Children's Hospital,calgary,48,51.074582,-114.148426
2,Devon General Hospital,devon,48,53.351493,-113.730785
3,Kimbelee Place,surrey,59,49.177418,-122.864118
4,Kimberley Special Care Home,kimberley,59,49.689998,-115.989647


In [19]:
alberta_hospitals = odhf[
    (odhf["Pruid"].astype(str) == "48")
    & (odhf["odhf_facility_type"].str.lower() == "hospitals")
]

edmonton_area = alberta_hospitals[
    alberta_hospitals["city"].str.lower().isin(["edmonton", "st. albert", "st albert"])
]

edmonton_area[["facility_name", "city", "latitude", "longitude"]]

,facility_name,city,latitude,longitude
84,East Edmonton Health Centre,edmonton,NaN,NaN
94,Alberta Hip and Knee Clinic,edmonton,NaN,NaN
198,Alberta Hospital Edmonton,edmonton,53.636387,-113.374023
390,Glenrose Rehabilitation Hospital,edmonton,53.560101,-113.497653
419,Grey Nuns Community Hospital,edmonton,53.461583,-113.429724
831,Mazankowski Alberta Heart Institute,edmonton,53.520191,-113.522727
864,Misericordia Community Hospital,edmonton,53.521028,-113.612320
881,Northeast Community Health Centre,edmonton,53.604272,-113.417472
1335,Royal Alexandra Hospital,edmonton,53.557387,-113.497772
1652,Sturgeon Community Hospital,st albert,53.655240,-113.625711


In [20]:
keep = { # our main focus right now
    "University of Alberta Hospital": "University of Alberta Hospital (incl. Stollery)",
    "Royal Alexandra": "Royal Alexandra Hospital",
    "Grey Nuns": "Grey Nuns Community Hospital",
    "Misericordia": "Misericordia Community Hospital",
    "Sturgeon": "Sturgeon Community Hospital",
}
# long term facilities and clinics would not fit well in the "wait-time" criteria we want to calculate so they are dropped
# plus they dont have valid coordinates

edmonton_hospitals = edmonton_area[
    edmonton_area["facility_name"].str.contains("|".join(keep), case=False)
].copy()

# give each hospital a clean, consistent name
edmonton_hospitals["hospital"] = edmonton_hospitals["facility_name"].apply(
    lambda name: next(label for key, label in keep.items() if key.lower() in name.lower())
)

edmonton_hospitals[["hospital", "latitude", "longitude"]]

,hospital,latitude,longitude
419,Grey Nuns Community Hospital,53.461583,-113.429724
864,Misericordia Community Hospital,53.521028,-113.612320
1335,Royal Alexandra Hospital,53.557387,-113.497772
1652,Sturgeon Community Hospital,53.655240,-113.625711
1929,University of Alberta Hospital (incl. Stollery),53.520634,-113.524527


In [21]:
waits_file = next((RAW / "cihi_waits").glob("*.xlsx"))
print(waits_file.name)

waits_book = pd.ExcelFile(waits_file)
print(waits_book.sheet_names)

wait-times-priority-procedures-in-canada-2008-2025-data-tables-en.xlsx
['Instructions', 'Table 1', 'Methodology notes', 'Contact information']


In [22]:
instructions = pd.read_excel(waits_file, sheet_name="Instructions", header=None)
for text in instructions[0].dropna(): # reads every non empty line
    print(text)
    print()

This workbook has 4 worksheets, including this Instructions page and 1 data table worksheet on tab 2. Notes and sources can be found after the table in column A. Methodology notes can be found on tab 3. Contact information can be found on tab 4.

Instructions
The following Unicode symbol is used in this workbook (this symbol may need to be enabled if using screen reading software):
asterisk (*)

Wait Times for Priority Procedures in Canada, 2008 to 2025 — Data Tables

This indicator provides wait times and volumes for priority procedures between 2008 and 2025. Wait times and volumes are provided at national, provincial and regional levels for 2 procedures: hip replacement and knee replacement. 

Wait times for hip fracture repair are calculated using the Discharge Abstract Database (DAD) and National Ambulatory Care Reporting System (NACRS) at the Canadian Institute for Health Information (CIHI). For all other procedures, aggregate wait time results are submitted to CIHI by provincial 

In [23]:
raw_table = pd.read_excel(waits_file, sheet_name="Table 1", header=None)
print(raw_table.shape)
raw_table.head(15)

(20511, 11)


,0,1,2,3,4,5,6,7,8,9,10
0,"Table 1 Wait times for priority procedures, b...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Reporting level,Province,Region,Indicator,Metric,Data year,Unit of measurement,Indicator result,NaN,NaN,NaN
2,Provincial,Alberta,NaN,Bladder Cancer Surgery,50th percentile,2008,Days,NaN,NaN,NaN,NaN
3,Provincial,Alberta,NaN,Bladder Cancer Surgery,90th percentile,2008,Days,NaN,NaN,NaN,NaN
4,Provincial,Alberta,NaN,Bladder Cancer Surgery,Volume,2008,Number of cases,NaN,NaN,NaN,NaN
5,Provincial,Alberta,NaN,Breast Cancer Surgery,50th percentile,2008,Days,NaN,NaN,NaN,NaN
6,Provincial,Alberta,NaN,Breast Cancer Surgery,90th percentile,2008,Days,NaN,NaN,NaN,NaN
7,Provincial,Alberta,NaN,Breast Cancer Surgery,Volume,2008,Number of cases,NaN,NaN,NaN,NaN
8,Provincial,Alberta,NaN,CABG,% meeting benchmark,2008,Proportion,99,NaN,NaN,NaN
9,Provincial,Alberta,NaN,CABG,50th percentile,2008,Days,7,NaN,NaN,NaN


In [24]:
raw_table.iloc[:, 0].dropna().unique()[:40] # first 40 distinct values in the first column

<StringArray>
[                                                                                                'Table 1  Wait times for priority procedures, by province and Canada, 2008 to 2025',
                                                                                                                                                                   'Reporting level',
                                                                                                                                                                        'Provincial',
                                                                                                                                                                          'National',
                                                                                                                                                                          'Regional',
                                                                            

In [25]:
waits = pd.read_excel(waits_file, sheet_name="Table 1", header=1, usecols="A:H") # keep only the first 8 cols
waits.columns = ["level", "province", "region", "indicator", "metric", "data_year", "unit", "result"]

# footnotes at the bottom have no indicator, drop them
waits = waits.dropna(subset=["indicator"])

regional = waits[waits["level"] == "Regional"]
print(regional.shape)
regional.groupby(["indicator", "metric"]).size()

(5576, 8)


indicator         metric             
Hip Replacement   % meeting benchmark    697
                  50th percentile        697
                  90th percentile        697
                  Volume                 697
Knee Replacement  % meeting benchmark    697
                  50th percentile        697
                  90th percentile        697
                  Volume                 697
dtype: int64

In [26]:
print(sorted(regional["data_year"].astype(str).unique()))
regional[regional["province"] == "Alberta"]["region"].unique()

['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


<StringArray>
[          'South Zone',         'Calgary Zone', 'Central Zone (Alta.)',
        'Edmonton Zone',   'North Zone (Alta.)']
Length: 5, dtype: str

notes- Alberta uses old 5 zones here, region names repeat across provinces so we can use codes to match them.

In [27]:
hip_2024 = regional[
    (regional["indicator"] == "Hip Replacement")
    & (regional["metric"] == "50th percentile")
    & (regional["data_year"].astype(str) == "2024")
].copy()

hip_2024["result"] = pd.to_numeric(hip_2024["result"], errors="coerce")

hip_2024.groupby("province").agg(
    regions=("region", "nunique"),
    with_data=("result", "count"),
)

,regions,with_data
province,,
Alberta,5,5
British Columbia,5,5
Manitoba,5,3
New Brunswick,2,2
Newfoundland and Labrador,5,5
Nova Scotia,4,4
Ontario,6,6
Prince Edward Island,1,1
Quebec,18,15


In [28]:
hip = regional[
    (regional["indicator"] == "Hip Replacement")
    & (regional["data_year"].astype(str) == "2024") # hip replacement rows for 2024
].copy()
hip["result"] = pd.to_numeric(hip["result"], errors="coerce") # convert to numbers

hip_waits = hip.pivot_table(index=["province", "region"], columns="metric", values="result").reset_index()
hip_waits.columns.name = None
hip_waits = hip_waits.rename(columns={ # instead of region having 4 rows- it one row with 4 cols now
    "50th percentile": "median_wait_days",
    "90th percentile": "p90_wait_days",
    "% meeting benchmark": "pct_meeting_benchmark",
    "Volume": "volume",
})
hip_waits = hip_waits.dropna(subset=["median_wait_days"]) # drops median with no wait days

hip_waits.to_csv(ROOT / "data" / "processed" / "hip_waits_2024.csv", index=False)
print(hip_waits.shape)
hip_waits.sort_values("median_wait_days", ascending=False).head(10)

(51, 6)


,province,region,pct_meeting_benchmark,median_wait_days,p90_wait_days,volume
16,Newfoundland and Labrador,Eastern–Rural Zone,24.000000,422.0,757.0,46.0
8,British Columbia,Northern Health,35.700000,363.0,754.0,280.0
44,Quebec,Outaouais Region,23.400000,345.0,495.0,167.0
48,Saskatchewan,Regina Zone,40.322581,292.0,583.0,434.0
39,Quebec,Laurentides Region,31.000000,286.0,595.0,409.0
33,Quebec,Capitale-Nationale Region,31.600000,280.0,382.5,370.0
10,Manitoba,Prairie Mountain Health Region,29.000000,278.0,349.0,49.0
50,Saskatchewan,South West Zone,27.272727,262.5,377.0,44.0
31,Quebec,Abitibi-Témiscamingue Region,44.700000,258.0,593.0,103.0
17,Newfoundland and Labrador,Eastern–Urban Zone,33.000000,252.0,748.0,147.0


In [29]:
hip_waits[hip_waits["province"] == "Alberta"] # alberta metrics

,province,region,pct_meeting_benchmark,median_wait_days,p90_wait_days,volume
0,Alberta,Calgary Zone,75.1,108.0,278.0,935.0
1,Alberta,Central Zone (Alta.),76.3,98.0,288.0,287.0
2,Alberta,Edmonton Zone,79.2,95.0,257.0,903.0
3,Alberta,North Zone (Alta.),52.6,173.0,455.0,287.0
4,Alberta,South Zone,64.3,115.0,383.0,350.0


In [30]:
hip_waits["median_wait_days"].describe() # across all regions

count     51.000000
mean     169.941176
std       83.513451
min       31.000000
25%      107.500000
50%      156.000000
75%      213.500000
max      422.000000
Name: median_wait_days, dtype: float64

In [31]:
hip_waits["rank_shortest"] = hip_waits["median_wait_days"].rank(method="min").astype(int)
hip_waits[hip_waits["region"] == "Edmonton Zone"][["region", "median_wait_days", "rank_shortest"]]

,region,median_wait_days,rank_shortest
2,Edmonton Zone,95.0,7


### What the wait data shows (hip replacement, 2024)

- Across 51 regions, a typical wait is about 5 months, ranging from about 1 month to over a year.
- The longest waits are mostly in rural and remote regions.
- Edmonton waits about 3 months, 7th shortest of 51, and is the best in Alberta.
- This is the "what" and not exactly the "why"

In [32]:
phys_file = next((RAW / "cihi_physicians").glob("*.xlsx"))
print(phys_file.name)
print(pd.ExcelFile(phys_file).sheet_names)

supply-distribution-migration-physicians-in-canada-2024-data-tables-en.xlsx
['Supply', 'Instructions ', 'Table of contents ', 'Table A Summary', 'Table B Jurisdiction profile', 'Sheet1', 'Table C Health regions ', 'Table 1.0–1.2 ', 'Table 2.0–2.2 ', 'Table 3.0 ', 'Table 4.0', 'Table 5.0–5.1', 'Table 6.0', 'Table 7.0–7.1', 'Table 8.0', 'Table 9.0', 'Table 10.0', 'Table 11.0', 'Table 12.0', 'Table 13.0', 'Table 14.0–14.1', 'Table 15.0', 'Table 16.0', 'Table 17.0–17.1', 'Table 18.0', 'Table 19.0', 'Table 20.0', 'Table 21.0', 'Table 22.0', 'Table 22.1', 'Table 22.2', 'Table 23.0', 'Table 23.1', 'Table 23.2', 'Table A1', 'Methodology ', 'Contact information']


In [33]:
phys_raw = pd.read_excel(phys_file, sheet_name="Table C Health regions ", header=None)
print(phys_raw.shape)
phys_raw.head(20)

(113, 21)


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
0,Table C Profile of family medicine physicians...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Jurisdiction,Map code: Health region,Name: Health region,"Total number of family medicine physicians, 2020","Total number of family medicine physicians, 2024",Percentage change in family medicine physician...,"Family medicine physicians per 100,000 populat...","Average age of family medicine physicians,* \n...","Percentage female family medicine physicians,†...",Percentage Canadian-trained family medicine ph...,...,"Total number of specialists, 2024","Percentage change in specialists, \n2020 to 2024","Specialist physicians per 100,000 population, ...","Average age of specialists,* \n2024","Percentage female specialists,† \n2024","Percentage Canadian-trained specialists,‡ \n2024",NaN,NaN,NaN,NaN
2,Canada,—,Canada,46797,49188,0.051093,119.13216,49.364319,0.507258,0.684,...,50367,0.109992,121.98767,49.523011,0.411559,0.765268,NaN,49.523011,0.411559,0.765268
3,N.L.,1000,Newfoundland and Labrador,684,662,-0.032164,121.412864,47.337368,0.503021,0.722054,...,759,0.121123,139.202967,48.894459,0.40975,0.719368,1.0,48.894459,0.409750,0.719368
4,NaN,1020,Eastern–Urban Zone,328,335,0.021341,142.765821,47.492537,0.6,0.850746,...,516,0.146667,219.901982,48.434109,0.449612,0.843023,NaN,48.434109,0.449612,0.843023
5,NaN,1021,Eastern–Rural Zone,110,88,-0.2,81.804153,50.215909,0.420455,0.659091,...,50,-0.074074,46.479633,49.244898,0.3,0.44,NaN,49.244898,0.300000,0.440000
6,NaN,1022,Central Zone,108,94,-0.12963,103.298974,46.904255,0.351064,0.489362,...,84,0.183099,92.309721,49.428571,0.309524,0.511905,NaN,49.428571,0.309524,0.511905
7,NaN,1023,Western Zone,90,105,0.166667,138.58276,46.209524,0.409524,0.647619,...,80,0.052632,105.586865,48.7125,0.4125,0.55,NaN,48.712500,0.412500,0.550000
8,NaN,1024,Labrador–Grenfell Zone,46,39,-0.152174,107.562469,43.578947,0.487179,0.538462,...,27,0.125,74.466325,55.37037,0.185185,0.037037,NaN,55.370370,0.185185,0.037037
9,NaN,1099,Unknown,2,1,§,NaN,44,0,0,...,2,§,NaN,56.5,0,0.5,NaN,56.500000,0.000000,0.500000


In [34]:
phys = pd.read_excel(phys_file, sheet_name="Table C Health regions ", header=1)
for i, name in enumerate(phys.columns):
    print(i, name) # col names ans its positions

0 Jurisdiction
1 Map code: Health region
2 Name: Health region
3 Total number of family medicine physicians, 2020
4 Total number of family medicine physicians, 2024
5 Percentage change in family medicine physicians, 
2020 to 2024
6 Family medicine physicians per 100,000 population, 
2024
7 Average age of family medicine physicians,* 
2024
8 Percentage female family medicine physicians,† 
2024 
9 Percentage Canadian-trained family medicine physicians,‡  
2024
10 Total number of specialists, 2020
11 Total number of specialists, 2024
12 Percentage change in specialists, 
2020 to 2024
13 Specialist physicians per 100,000 population, 
2024
14 Average age of specialists,* 
2024
15 Percentage female specialists,† 
2024
16 Percentage Canadian-trained specialists,‡  
2024
17 Unnamed: 17
18 Unnamed: 18
19 Unnamed: 19
20 Unnamed: 20


In [35]:
code_col = phys.columns[1]
phys[phys[code_col].astype(str).str.startswith("48")].iloc[:, [0, 1, 2, 4, 6]] # alberta regions

,Jurisdiction,Map code: Health region,Name: Health region,"Total number of family medicine physicians, 2024","Family medicine physicians per 100,000 population, \n2024"
69,Alta.,4800,Alberta,5734.0,117.290344
70,NaN,4831,South Zone,343.0,104.326984
71,NaN,4832,Calgary Zone,2580.0,131.529845
72,NaN,4833,Central Zone,538.0,108.431015
73,NaN,4834,Edmonton Zone,1822.0,111.828469
74,NaN,4835,North Zone,422.0,89.223795
75,NaN,4899,Unknown,29.0,NaN
